# Landing-Pad → IMX500 (.rpk) für die Raspberry Pi AI Camera

Modell: Run F, eine Klasse `landingPad`.

Dieses Notebook macht Schritt 1 und 2. Schritt 3 braucht ein aarch64-Linux —
der Paketierer ist ein ARM-Binary und läuft auf Colab nicht. Dafür: der
Raspberry Pi oder der GitHub-Actions-Workflow.

> **Nicht „Alle ausführen" benutzen.** In Abschnitt 2 wird die Laufzeit einmal
> absichtlich neu gestartet. Führe die Zellen der Reihe nach aus und mach nach
> dem Neustart bei Abschnitt 3 weiter.

## 1. `imx_bundle.zip` hochladen

In [ ]:
from google.colab import files

up = files.upload()          # imx_bundle.zip auswählen

In [ ]:
!mkdir -p /content/imx
!cd /content/imx && unzip -oq /content/imx_bundle.zip
!ls /content/imx /content/imx/data

## 2. Abhängigkeiten — und einmal neu starten

Zwei Stolpersteine, beide hier abgeräumt:

**TensorFlow muss weg.** Sonys Konverter verlangt `protobuf==4.25.5`, das in
Colab vorinstallierte TensorFlow braucht aber protobuf 5.x. Ultralytics stuft
protobuf beim Export herunter, danach ist TensorFlow kaputt und der Export
stirbt mit `cannot import name 'runtime_version'`. Für unseren PyTorch-Weg wird
TensorFlow gar nicht gebraucht — `model_compression_toolkit` importiert es nur,
weil es da ist.

**Die Pakete vorher installieren**, nicht erst beim Export. Sonst tauscht
Ultralytics mitten im Lauf Abhängigkeiten aus, die längst importiert sind.

In [ ]:
!pip -q uninstall -y tensorflow tensorflow-cpu tf-keras keras 2>/dev/null
!pip -q install ultralytics
!pip -q install "model-compression-toolkit>=2.4.1" "edge-mdt-cl<1.1.0" \
                "edge-mdt-tpc>=1.2.0" "pydantic<=2.11.7" \
                "imx500-converter[pt]>=3.17.3"
!apt-get -qq update && apt-get -qq install -y default-jre > /dev/null
print("fertig — jetzt die nächste Zelle für den Neustart")

### Laufzeit neu starten

Die Zelle unten beendet die Laufzeit absichtlich („Sitzung abgestürzt" ist hier
die erwartete Meldung). `/content` bleibt erhalten, das Bundle muss **nicht**
neu hochgeladen werden. Danach unten bei Abschnitt 3 weitermachen.

In [ ]:
import os

os.kill(os.getpid(), 9)

## 3. Export  ← hier nach dem Neustart weitermachen

Gradientenbasiertes Post-Training-Quantisieren über die Kalibrierbilder, keine
einfache INT8-Umwandlung — deshalb dauert es Minuten und deshalb `data=`.

`data=calib.yaml` zeigt auf 276 Kalibrierbilder. Ultralytics zieht die
Kalibrierung aus dem *val*-Eintrag, und der reguläre val-Split hat nur 25
Bilder — zu wenig, der Export warnt dann selbst („>300 images recommended").

`imgsz=320` ist die Trainingsgröße von Run F. Nicht ändern: außerhalb seiner
Trainingsauflösung stiegen die Fehlalarme in den Messungen von 0.00 auf über 1
pro Bild.

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/imx/best.pt")
print("Klassen:", model.names)          # erwartet: {0: 'landingPad'}

In [ ]:
out = model.export(
    format="imx",
    imgsz=320,
    int8=True,
    data="/content/imx/data/calib.yaml",
)
print("Export-Verzeichnis:", out)

In [ ]:
!ls -la /content/imx/best_imx_model

## 4. Gegenprüfen

Referenzwert zuerst, damit du auch dann eine Zahl hast, wenn Ultralytics das
IMX-Verzeichnis nicht direkt validieren kann. Hier wird `data.yaml` benutzt,
also der echte, zurückgehaltene val-Split — nicht das Kalibrierset.

In [ ]:
fp32 = YOLO("/content/imx/best.pt").val(
    data="/content/imx/data/data.yaml", imgsz=320, split="val", verbose=False)
print(f"Referenz FP32: mAP50={fp32.box.map50:.4f}  mAP50-95={fp32.box.map:.4f}")

In [ ]:
try:
    q = YOLO("/content/imx/best_imx_model", task="detect")
    r = q.val(data="/content/imx/data/data.yaml", imgsz=320,
              split="val", verbose=False)
    print(f"quantisiert:   mAP50={r.box.map50:.4f}  mAP50-95={r.box.map:.4f}")
except Exception as e:
    print("IMX-Modell hier nicht validierbar:", type(e).__name__, e)
    print("-> dann erst auf dem Pi im Livebild prüfen")

## 5. `packerOut.zip` herunterladen

In [ ]:
from google.colab import files

files.download("/content/imx/best_imx_model/packerOut.zip")

## 6. Schritt 3 — `network.rpk` bauen

**Auf dem Raspberry Pi:**

```bash
sudo apt update && sudo apt install -y imx500-tools
imx500-package -i packerOut.zip -o .
```

**Oder ohne Pi:** `packerOut.zip` ins GitHub-Repo legen und den Workflow
*IMX500 .rpk bauen* starten (kostenloser `ubuntu-24.04-arm`-Runner), dann
`network.rpk` als Artefakt herunterladen.

Für die Kamera brauchst du neben der `.rpk` eine Labels-Datei mit der einen
Zeile `landingPad`; in der Post-Processing-JSON zeigt `network_file` auf die
`.rpk`.